# Meridian Financial — Fraud Model v1

Fresh-reservation-safe baseline model notebook.

This version:
- finds the intended `fraud-detection` repository instead of assuming it lives directly under `/opt/app-root/src`
- requires the repository to contain the expected training CSVs
- refuses to continue if multiple valid repo copies are found
- writes the v1 ONNX artifact inside the same PVC-backed repository at `models/fraud/1/model.onnx`
- validates the ONNX artifact before deployment

The model expects these five raw features in this exact order:

1. `distance_from_last_transaction`
2. `ratio_to_median_purchase_price`
3. `used_chip`
4. `used_pin_number`
5. `online_order`

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import sklearn
import onnx

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

def is_repo_root(path: Path) -> bool:
    return (
        (path / "data" / "train.csv").is_file()
        and (path / "data" / "validate.csv").is_file()
        and (path / "data" / "test.csv").is_file()
    )

cwd = Path.cwd().resolve()
candidates = []

# Prefer the directory the notebook was opened from.
if is_repo_root(cwd):
    candidates.append(cwd)

# Walk upward if Jupyter started in a child directory.
for parent in cwd.parents:
    if is_repo_root(parent):
        candidates.append(parent)

# Fresh-reservation fallback: the repo should normally be inside the
# workbench PVC mount somewhere under /opt/app-root/src/<pvc-mount>/fraud-detection.
search_root = Path("/opt/app-root/src")
if search_root.exists():
    for candidate in search_root.glob("*/fraud-detection"):
        candidate = candidate.resolve()
        if is_repo_root(candidate):
            candidates.append(candidate)

# De-duplicate.
unique_candidates = []
for candidate in candidates:
    if candidate not in unique_candidates:
        unique_candidates.append(candidate)

if not unique_candidates:
    raise FileNotFoundError(
        "Could not locate the fraud-detection repository. "
        "Clone it onto the workbench PVC, open this notebook from that repo, and rerun."
    )

if len(unique_candidates) > 1:
    raise RuntimeError(
        "Multiple valid fraud-detection repositories were found. "
        "Remove the stale copy or open this notebook from the intended PVC-backed repo.\nFound:\n"
        + "\n".join(str(p) for p in unique_candidates)
    )

REPO_DIR = unique_candidates[0]
DATA_DIR = REPO_DIR / "data"
MODEL_DIR = REPO_DIR / "models" / "fraud" / "1"
ARTIFACT_DIR = REPO_DIR / "artifact"

os.chdir(REPO_DIR)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_DIR)
print("Working directory:", Path.cwd())
print("v1 output:", MODEL_DIR / "model.onnx")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("onnx:", onnx.__version__)

## Verify the dataset is present

In [ ]:
required_files = [
    DATA_DIR / "train.csv",
    DATA_DIR / "validate.csv",
    DATA_DIR / "test.csv",
]

for f in required_files:
    print(f"{f}: {f.exists()}")

missing = [str(f) for f in required_files if not f.exists()]
if missing:
    raise FileNotFoundError(
        "Missing expected dataset files:\n" + "\n".join(missing)
    )

## Load the data

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "validate.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

## Select features and target

In [ ]:
FEATURES = [
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "used_chip",
    "used_pin_number",
    "online_order",
]

target_candidates = ["fraud", "is_fraud", "label", "target", "Class"]
TARGET = next((c for c in target_candidates if c in train_df.columns), None)

if TARGET is None:
    raise ValueError(
        "Could not identify the target column. "
        f"Available columns: {list(train_df.columns)}"
    )

missing_features = [c for c in FEATURES if c not in train_df.columns]
if missing_features:
    raise ValueError(f"Missing expected feature columns: {missing_features}")

X_train = train_df[FEATURES].astype("float32")
y_train = train_df[TARGET].astype(int)

X_val = val_df[FEATURES].astype("float32")
y_val = val_df[TARGET].astype(int)

X_test = test_df[FEATURES].astype("float32")
y_test = test_df[TARGET].astype(int)

print("Target:", TARGET)
print("Features:", FEATURES)
print("Training fraud rate:", f"{y_train.mean():.2%}")

## Train the model

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="lbfgs",
        random_state=42,
    )),
])

model.fit(X_train, y_train)

print("Model trained successfully.")

## Evaluate

In [ ]:
def evaluate(model, X, y, name):
    pred = model.predict(X)
    prob = model.predict_proba(X)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, prob),
    }

    print(name)
    for k, v in metrics.items():
        print(f"  {k:10s}: {v:.4f}")

    return metrics

val_metrics = evaluate(model, X_val, y_val, "Validation")
test_metrics = evaluate(model, X_test, y_test, "Test")

## Test the known Meridian normal transaction

In [ ]:
meridian_normal = pd.DataFrame([{
    "distance_from_last_transaction": 0.3111400080477545,
    "ratio_to_median_purchase_price": 1.9459399775518593,
    "used_chip": 1.0,
    "used_pin_number": 0.0,
    "online_order": 0.0,
}], columns=FEATURES).astype("float32")

normal_prob = model.predict_proba(meridian_normal)[0, 1]
normal_label = model.predict(meridian_normal)[0]

print("Label:", int(normal_label))
print("Fraud probability:", f"{normal_prob:.4%}")

## Export the complete pipeline to ONNX

In [ ]:
initial_type = [
    ("float_input", FloatTensorType([None, len(FEATURES)]))
]

onnx_model = convert_sklearn(
    model,
    initial_types=initial_type,
    options={id(model.named_steps["classifier"]): {"zipmap": False}},
    target_opset=17,
)

model_path = MODEL_DIR / "model.onnx"
model_path.write_bytes(onnx_model.SerializeToString())

print("Saved:", model_path)

## Validate the ONNX artifact

In [ ]:
loaded = onnx.load(model_path)
onnx.checker.check_model(loaded)

print("ONNX validation passed.")
print("Model size:", model_path.stat().st_size, "bytes")

## Save model metadata

In [ ]:
metadata = {
    "model_version": 1,
    "model_type": "StandardScaler + LogisticRegression",
    "features": FEATURES,
    "target": TARGET,
    "artifact": str(model_path),
    "notes": "Baseline Meridian Financial fraud model",
}

metadata_path = ARTIFACT_DIR / "model_v1_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2))

print(metadata_path.read_text())

## Final verification

In [ ]:
print("Expected model location:")
print(model_path)

print("\nFile exists:", model_path.exists())

if not model_path.exists():
    raise FileNotFoundError(f"Model artifact was not created at {model_path}")

print("\nReady for OpenShift AI deployment.")

## Deploy in OpenShift AI

Before deployment, verify the printed absolute model path is inside the **PVC-backed**
`fraud-detection` repository and ends with:

```text
fraud-detection/models/fraud/1/model.onnx
```

Because the repository itself is stored directly on the workbench PVC, use this
PVC-relative model path in OpenShift AI:

```text
fraud-detection/models/fraud/1
```

Keep:

```text
MLSERVER_MODEL_URI=/mnt/models/model.onnx
```